In [1]:
import ast
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from tabulate import tabulate

In [2]:
class PairwiseResult:
    unparsed: int = 0
    ties: int = 0
    win1: int = 0
    win2: int = 0


def get_pairwise_results(p: Path) -> PairwiseResult:
    out = PairwiseResult()

    with p.open(newline="") as f:
        r = csv.DictReader(f)
        for line in r:
            res = int(line["choice"])
            match res:
                case -1:
                    out.unparsed += 1
                case 0:
                    out.ties += 1
                case 1:
                    out.win1 += 1
                case 2:
                    out.win2 += 1

    return out

In [3]:
datasets = ["lcstep", "recipenlg", "champ"]
totals = {
    "lcstep": 27,
    "recipenlg": 100,
    "champ": 27,
}
baseline_emb = "hf-all-mpnet-base-v2"
embedders = [
    "hf-nomic-embed-text-v1.5",
    "hf-gte-large-en-v1.5",
    "openai-text-embedding-3-large",
]
systems = ["aag"]

output_dir = Path("../output")

In [4]:
table = {
    "system@dataset": [f"{s}@{d}" for s in systems for d in datasets],
    "total": [totals[d] for s in systems for d in datasets],
}
headers = ["system@dataset", "total"]
for emb in embedders:
    improve_count = []
    tie_count = []
    unparsed_count = []

    for system in systems:
        for dataset in datasets:
            res = get_pairwise_results(output_dir / f"{system}_{dataset}_{baseline_emb}_{emb}_pair_eval.csv")

            improve_count.append(res.win2 - res.win1)
            tie_count.append(res.ties)
            unparsed_count.append(res.unparsed)

    table[emb] = improve_count
    headers.append(emb)
    if sum(unparsed_count) > 0:
        table["unparsed_" + emb] = unparsed_count
        headers.append("unparsed")
    table["ties_" + emb] = tie_count
    headers.append("ties")

print(tabulate(table, headers=headers, tablefmt="grid"))

+------------------+---------+----------------------------+--------+------------------------+--------+---------------------------------+--------+
| system@dataset   |   total |   hf-nomic-embed-text-v1.5 |   ties |   hf-gte-large-en-v1.5 |   ties |   openai-text-embedding-3-large |   ties |
+==================+=========+============================+========+========================+========+=================================+========+
| aag@lcstep       |      27 |                          3 |      2 |                      1 |      6 |                               4 |      3 |
+------------------+---------+----------------------------+--------+------------------------+--------+---------------------------------+--------+
| aag@recipenlg    |     100 |                         16 |     14 |                      4 |     12 |                               6 |      8 |
+------------------+---------+----------------------------+--------+------------------------+--------+----------------------

In [7]:
embedders = [
    "hf-all-mpnet-base-v2",
    "hf-nomic-embed-text-v1.5",
    "hf-gte-large-en-v1.5",
    "openai-text-embedding-3-large",
]

table = {
    "dataset": datasets,
    "total": [totals[d] for d in datasets],
}
headers = ["dataset", "total"]
for emb in embedders:
    improve_count = []
    tie_count = []
    unparsed_count = []

    for dataset in datasets:
        res = get_pairwise_results(output_dir / f"rag_aag_{dataset}_{emb}_pair_eval.csv")

        improve_count.append(res.win2 - res.win1)
        tie_count.append(res.ties)
        unparsed_count.append(res.unparsed)

    table[emb] = improve_count
    headers.append(emb)
    if sum(unparsed_count) > 0:
        table["unparsed_" + emb] = unparsed_count
        headers.append("unparsed")
    table["ties_" + emb] = tie_count
    headers.append("ties")

print(tabulate(table, headers=headers, tablefmt="grid"))

+-----------+---------+------------------------+--------+----------------------------+--------+------------------------+--------+---------------------------------+--------+
| dataset   |   total |   hf-all-mpnet-base-v2 |   ties |   hf-nomic-embed-text-v1.5 |   ties |   hf-gte-large-en-v1.5 |   ties |   openai-text-embedding-3-large |   ties |
+===========+=========+========================+========+============================+========+========================+========+=================================+========+
| lcstep    |      27 |                     -2 |      7 |                          1 |      6 |                      0 |      7 |                               4 |      5 |
+-----------+---------+------------------------+--------+----------------------------+--------+------------------------+--------+---------------------------------+--------+
| recipenlg |     100 |                    -32 |      9 |                        -25 |     12 |                    -21 |     14 |      

In [8]:
table = {
    "dataset": datasets,
    "total": [totals[d] for d in datasets],
}
headers = ["dataset", "total"]
improve_count = []
tie_count = []
unparsed_count = []

for dataset in datasets:
    res = get_pairwise_results(output_dir / f"rag_summ-critic_{dataset}_hf-all-mpnet-base-v2_pair_eval.csv")

    improve_count.append(res.win2 - res.win1)
    tie_count.append(res.ties)
    unparsed_count.append(res.unparsed)

table[emb] = improve_count
headers.append(emb)
if sum(unparsed_count) > 0:
    table["unparsed_" + emb] = unparsed_count
    headers.append("unparsed")
table["ties_" + emb] = tie_count
headers.append("ties")

print(tabulate(table, headers=headers, tablefmt="grid"))

+-----------+---------+---------------------------------+--------+
| dataset   |   total |   openai-text-embedding-3-large |   ties |
+===========+=========+=================================+========+
| lcstep    |      27 |                               8 |      9 |
+-----------+---------+---------------------------------+--------+
| recipenlg |     100 |                              17 |     10 |
+-----------+---------+---------------------------------+--------+
| champ     |      27 |                               4 |      5 |
+-----------+---------+---------------------------------+--------+


### Eval Heuristics

In [ ]:
for sys in system:
    for data in dataset:
        for embed in embedder:
            eval_results_file = f"../output/{sys}/{data}/{embed}/eval_results.csv"
            eval_metrics = pd.read_csv(eval_results_file, header=0)
            eval_metrics = eval_metrics.drop(eval_metrics[eval_metrics.Overall < 0].index)
            if 'ROUGE' in list(eval_metrics):
                eval_metrics = eval_metrics.drop(columns=['ROUGE'])
            print(f"Printing mean stats for output/{sys}/{data}/{embed}: {eval_metrics.mean()}")

In [10]:
for sys in system:
    for data in dataset:
        for embed in embedder:
            eval_results_file = f"../output/{sys}/{data}/{embed}/eval_results.csv"
            eval_metrics = pd.read_csv(eval_results_file, header=0).drop(columns=['_id'])
            eval_metrics = eval_metrics.drop(eval_metrics[eval_metrics.Overall < 0].index)
            if 'ROUGE' in list(eval_metrics):
                eval_metrics['ROUGE'] = eval_metrics['ROUGE'].apply(lambda x: ast.literal_eval(x))
                rouge_df = eval_metrics['ROUGE'].apply(pd.Series)
                rouge_df = rouge_df[['rouge1_fmeasure', 'rouge2_fmeasure', 'rougeL_fmeasure', 'rougeLsum_fmeasure']]
                new_df = pd.concat([eval_metrics.drop(columns=["ROUGE"]), rouge_df], axis=1)
                print(f"Printing mean stats for output/{sys}/{data}/{embed}: {new_df.mean()}")
            else:
                print(f"Printing mean stats for output/{sys}/{data}/{embed}: {eval_metrics.mean()}")

Printing mean stats for output/rag/lcstep/hf-all-mpnet-base-v2: Api-Overlap           0.303395
TfIdf                 0.396296
Inp-Used              0.518519
Overall               4.222222
rouge1_fmeasure       0.016524
rouge2_fmeasure       0.000245
rougeL_fmeasure       0.014738
rougeLsum_fmeasure    0.016474
dtype: float64
Printing mean stats for output/rag/lcstep/openai-text-embedding-3-large: Api-Overlap           0.311199
TfIdf                 0.446296
Inp-Used              0.537037
Overall               4.296296
rouge1_fmeasure       0.020234
rouge2_fmeasure       0.000537
rougeL_fmeasure       0.017539
rougeLsum_fmeasure    0.020234
dtype: float64
Printing mean stats for output/rag/recipenlg/hf-all-mpnet-base-v2: TfIdf                 0.266000
Ing_Used              0.803304
Edit-Distance         9.260000
Num-Compare           0.057429
Overall               4.830000
rouge1_fmeasure       0.043918
rouge2_fmeasure       0.001305
rougeL_fmeasure       0.031102
rougeLsum_fmeasure    

In [ ]:
for ev_metric_name in list(eval_metrics.columns.values):
    if ev_metric_name == "_id" or ev_metric_name == "Overall":
        continue
    sp_corr = stats.spearmanr(eval_metrics[ev_metric_name], eval_metrics['Overall']/10)
    print(f"Spearman Coefficient of Overall Score and {ev_metric_name} is: {sp_corr.statistic}")

In [ ]:
plt.scatter(eval_metrics['Ing_Used'].to_numpy(), eval_metrics['Overall'].to_numpy())

### Pairwise Evals

In [15]:

##without GT steps - 5 runs panel strategy
# dataset = ["recipenlg"]
res = {"Datasets":dataset, "Total":[27,100,27]}
for emb in embedder:
    service, model = emb.split("-", maxsplit=1)
    emb_res = []
    emb_unparsed = []
    emb_tie = []
    for data in dataset:
        p_eval_file = f'../output/without-gt/rag_aag_{data}_{emb}_pair_eval.csv'
        df = pd.read_csv(p_eval_file, header=0, usecols=["question_id", "choice"])
        tot = len(df)
        choices_orig = df['choice'].to_numpy()
        df = df.drop(df[df.choice <= 0].index)
        choices = df['choice'].to_numpy()
        num_aag = np.sum(choices-1)
        num_rag = len(choices) - num_aag
        emb_res.append(num_aag-num_rag)
        emb_unparsed.append(np.sum(choices_orig < 0))
        emb_tie.append(np.sum(choices_orig == 0))
    res[emb] = emb_res
    res[f"unparsed_{service}"] = emb_unparsed
    res[f"ties_{service}"] = emb_tie

print(tabulate(res, headers="keys", tablefmt="grid"))

+------------+---------+------------------------+---------------+-----------+---------------------------------+-------------------+---------------+
| Datasets   |   Total |   hf-all-mpnet-base-v2 |   unparsed_hf |   ties_hf |   openai-text-embedding-3-large |   unparsed_openai |   ties_openai |
+============+=========+========================+===============+===========+=================================+===================+===============+
| lcstep     |      27 |                     -8 |             0 |         7 |                               4 |                 0 |             7 |
+------------+---------+------------------------+---------------+-----------+---------------------------------+-------------------+---------------+
| recipenlg  |     100 |                     -1 |             0 |        15 |                               7 |                 0 |             9 |
+------------+---------+------------------------+---------------+-----------+---------------------------------+-

In [3]:

##without GT steps - 5 runs panel strategy - between embeds
# dataset = ["recipenlg"]
res = {"Datasets":dataset, "Total":[27,100,27]}
for emb in embedder:
    service, model = emb.split("-", maxsplit=1)
    emb_res = []
    emb_unparsed = []
    emb_tie = []
    for data in dataset:
        p_eval_file = f'../output/without-gt-between-embeds/aag_aag_{data}_{emb}_pair_eval.csv'
        df = pd.read_csv(p_eval_file, header=0, usecols=["question_id", "choice"])
        tot = len(df)
        choices_orig = df['choice'].to_numpy()
        df = df.drop(df[df.choice <= 0].index)
        choices = df['choice'].to_numpy()
        num_aag = np.sum(choices-1)
        num_rag = len(choices) - num_aag
        emb_res.append(num_aag-num_rag)
        emb_unparsed.append(np.sum(choices_orig < 0))
        emb_tie.append(np.sum(choices_orig == 0))
    res["score"] = emb_res
    res[f"unparsed"] = emb_unparsed
    res[f"ties"] = emb_tie
    break

print(tabulate(res, headers="keys", tablefmt="grid"))

In [16]:

##with GT steps
res = {"Datasets":dataset, "Total":[27, 100, 27]}
for emb in embedder:
    service, model = emb.split("-", maxsplit=1)
    emb_res = []
    emb_unparsed = []
    for data in dataset:
        p_eval_file = f'../output/with-gt/rag_aag_{data}_{emb}_pair_eval.csv'
        df = pd.read_csv(p_eval_file, header=0, usecols=["question_id", "choice"])
        tot = len(df)
        df = df.drop(df[df.choice < 0].index)
        choices = df['choice'].to_numpy()
        num_aag = np.sum(choices-1)
        num_rag = len(choices) - num_aag
        emb_res.append(num_aag-num_rag)
        emb_unparsed.append(tot-len(choices))
    res[emb] = emb_res
    res[f"unparsed_{service}"] = emb_unparsed

print(tabulate(res, headers="keys", tablefmt="grid"))

+------------+---------+------------------------+---------------+---------------------------------+-------------------+
| Datasets   |   Total |   hf-all-mpnet-base-v2 |   unparsed_hf |   openai-text-embedding-3-large |   unparsed_openai |
+============+=========+========================+===============+=================================+===================+
| lcstep     |      27 |                     -3 |             2 |                              -4 |                 1 |
+------------+---------+------------------------+---------------+---------------------------------+-------------------+
| recipenlg  |     100 |                     -2 |             2 |                              20 |                 4 |
+------------+---------+------------------------+---------------+---------------------------------+-------------------+
| champ      |      27 |                      1 |             0 |                              -5 |                 0 |
+------------+---------+----------------